## Personal Chef

In [4]:
from json import tool

import dotenv

dotenv.load_dotenv()

True

In [8]:
from langchain.agents import create_agent
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel

tavily_client = TavilyClient()

class Recipe(BaseModel):
    name: str
    ingredients: list[str]
    steps: list[str]
    description: str
    servings: int

@tool
def web_search(query : str) -> Dict[str, Any]:
    """Search the web for information"""

    return tavily_client.search(query)

system_prompt = "You are a chef in 5 star restaurant. You excell in making Indian food. You should assist me cooking. You can use only the ingredients given to you(which i have)."

chef_agent = create_agent(
    model='google_genai:gemini-3.1-flash-lite',
    system_prompt = system_prompt,
    tools = [web_search],
    checkpointer = InMemorySaver(),
    response_format=Recipe
)

In [10]:
from langchain.messages import HumanMessage

question = HumanMessage(content="I have some left over Chicken and rice.")

config = {"configurable" : {"thread_id": 1}}

response = chef_agent.invoke(
    {'messages': [question],},
    config,
)

print(response["messages"][-1].content)

Deserializing unregistered type __main__.Recipe from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'Recipe')]


[{'type': 'text', 'text': '{\n  "name": "Quick Indian Spiced Chicken and Rice Skillet",\n  "ingredients": [\n    "Leftover cooked chicken, shredded or diced",\n    "Leftover cooked rice",\n    "1 tablespoon ghee or cooking oil",\n    "1/2 teaspoon cumin seeds",\n    "1 small onion, finely chopped",\n    "1 teaspoon ginger-garlic paste",\n    "1/2 teaspoon turmeric powder",\n    "1 teaspoon garam masala",\n    "Salt to taste",\n    "Fresh coriander leaves for garnish"\n  ],\n  "steps": [\n    "Heat the ghee or oil in a pan over medium heat.",\n    "Add cumin seeds and let them sizzle for a few seconds.",\n    "Add the chopped onion and sauté until it turns translucent and lightly golden.",\n    "Stir in the ginger-garlic paste and cook for one minute until the raw smell disappears.",\n    "Add turmeric powder and garam masala, stirring quickly to toast the spices without burning them.",\n    "Add the leftover chicken and toss to coat it thoroughly in the spice mixture.",\n    "Add the l